# Evaluación Sumativa 1 — Análisis exploratorio e inferencial
## Dataset: Framingham Heart Study (riesgo de enfermedad coronaria a 10 años)

**MCDI501:** Estadística Computacional para la Toma de Decisiones
**Fase 2:** Búsqueda y recopilación de información

| | |
|---|---|
| **Integrantes** | Brian Cerda · Kattia Espinoza · Karla Martínez · Christopher Ulloa |
| **Docente** | «nombre del docente» |
| **Fecha** | «dd/mm/aaaa» |
| **Repositorio** | «https://github.com/usuario/repositorio» |
| **Dataset** | *Logistic regression To predict heart disease* (Kaggle), archivo `framingham.csv` |

Este cuaderno acompaña nuestro informe técnico y contiene el código completo y reproducible. Sigue el mismo orden del
informe: preparación y calidad de los datos, análisis exploratorio, estimación de parámetros y pruebas de hipótesis.
Las tablas y figuras numeradas son las mismas que aparecen en el informe; los gráficos marcados como
*complementarios* apoyan el análisis, pero no se incluyen en él. La semilla se fija en 42 para que los resultados se
puedan reproducir.

## Introducción

Las enfermedades cardiovasculares son una de las principales causas de muerte en el mundo, y una parte importante de
ellas se puede prevenir si se identifica a tiempo a las personas en riesgo. Nuestro problema es de **decisión bajo
incertidumbre**: a partir de información que se puede medir hoy en un control de salud (edad, presión arterial,
colesterol, hábitos y antecedentes), buscamos determinar qué características se asocian a desarrollar enfermedad
coronaria (CHD) en los próximos 10 años, para decidir a quién priorizar en prevención.

**Pregunta de investigación:** ¿qué factores demográficos, conductuales y clínicos se asocian con el desarrollo de
enfermedad coronaria a 10 años, y en qué magnitud?

En esta fase caracterizamos los datos, estimamos parámetros de la población con intervalos de confianza y contrastamos
hipótesis sobre factores de riesgo concretos. Los resultados serán la base del modelo predictivo de las fases siguientes.

---
## 0. Configuración y carga de datos

### 0.1 Librerías

Importamos al inicio todas las librerías que usaremos, para que las dependencias del cuaderno queden visibles en un
solo lugar: `numpy` y `pandas` para manipular los datos, `scipy.stats` para distribuciones y pruebas, `statsmodels`
para intervalos de proporciones y el análisis estratificado, `matplotlib` y `seaborn` para los gráficos, y `pathlib`
para manejar rutas de archivos que funcionen igual en el computador de cada integrante.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from statsmodels.stats.proportion import proportion_confint
from statsmodels.stats.contingency_tables import StratifiedTable

print("Versiones -> pandas:", pd.__version__, "| numpy:", np.__version__)

Versiones -> pandas: 3.0.5 | numpy: 2.5.3


Las librerías se cargaron correctamente. Dejamos registro de las versiones porque, si un integrante obtiene un resultado
distinto, lo primero que conviene revisar es si trabaja con otra versión.

### 0.2 Parámetros del análisis y estilo de los gráficos

Definimos en un solo lugar la semilla, el nivel de significancia, el nivel de confianza y las rutas de trabajo. Así,
si necesitamos cambiar un parámetro, lo modificamos aquí y no en distintas partes del código, lo que evita
inconsistencias y garantiza la reproducibilidad. También fijamos una paleta propia: azul para las personas sin CHD y
naranjo para quienes la desarrollan, que mantenemos en todos los gráficos.

In [2]:
# --- Parámetros del análisis ---
SEMILLA = 42
ALFA = 0.05                  # nivel de significancia de las pruebas
NIVEL_CONFIANZA = 0.95       # nivel de confianza de los intervalos
RUTA_DATOS = Path("data") / "framingham.csv"
CARPETA_FIGURAS = Path("figuras")
CARPETA_TABLAS = Path("tablas")
CARPETA_FIGURAS.mkdir(exist_ok=True)
CARPETA_TABLAS.mkdir(exist_ok=True)

np.random.seed(SEMILLA)
Z_CRITICO = stats.norm.ppf(1 - (1 - NIVEL_CONFIANZA) / 2)   # 1,96 para 95 %

# --- Paleta y estilo propios ---
COLOR_SIN_CHD = "#2a78d6"    # azul: personas sin CHD a 10 años
COLOR_CON_CHD = "#eb6834"    # naranjo: personas con CHD a 10 años
COLOR_TEXTO = "#3d3d3a"
COLOR_REFERENCIA = "#8a8984"
PALETA_OBJETIVO = {0: COLOR_SIN_CHD, 1: COLOR_CON_CHD}

plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 200, "font.size": 10,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": "#b5b4ad", "axes.titleweight": "bold", "axes.titlesize": 11,
    "axes.titlelocation": "left", "axes.grid": True, "axes.axisbelow": True,
    "grid.color": "#e6e5e0", "grid.linewidth": 0.6,
    "text.color": COLOR_TEXTO, "axes.labelcolor": COLOR_TEXTO,
    "xtick.color": COLOR_TEXTO, "ytick.color": COLOR_TEXTO,
})
pd.set_option("display.float_format", lambda valor: f"{valor:,.3f}")


def guardar_figura(figura: plt.Figure, nombre: str) -> None:
    """Guarda la figura en la carpeta de figuras para usarla en el informe."""
    figura.savefig(CARPETA_FIGURAS / f"{nombre}.png", bbox_inches="tight")


def guardar_tabla(tabla: pd.DataFrame, nombre: str) -> None:
    """Exporta la tabla a CSV para que el informe use exactamente los resultados de esta ejecución."""
    tabla.to_csv(CARPETA_TABLAS / f"{nombre}.csv", encoding="utf-8")


print(f"Semilla: {SEMILLA} | α = {ALFA} | confianza = {NIVEL_CONFIANZA:.0%} | z crítico = {Z_CRITICO:.3f}")

Semilla: 42 | α = 0.05 | confianza = 95% | z crítico = 1.960


Quedaron definidos α = 0,05 y un nivel de confianza del 95 % (z crítico = 1,96). Los gráficos del informe se guardan
automáticamente en la carpeta `figuras/` y las tablas en la carpeta `tablas/` (formato CSV), de modo que el informe
se construye directamente con los resultados de esta ejecución.

### 0.3 Carga del dataset

Leemos el archivo y traducimos los nombres de las columnas al español. Los nombres originales vienen abreviados en
inglés (`sysBP`, `prevalentHyp`), lo que dificulta la lectura del código y de las tablas. Conservamos el diccionario de
equivalencias para mantener la trazabilidad con la fuente original.

In [3]:
NOMBRES_EN_ESPANOL = {
    "male": "sexo_masculino", "age": "edad", "education": "educacion",
    "currentSmoker": "fumador_actual", "cigsPerDay": "cigarrillos_dia",
    "BPMeds": "medicamento_presion", "prevalentStroke": "acv_previo",
    "prevalentHyp": "hipertension", "diabetes": "diabetes",
    "totChol": "colesterol_total", "sysBP": "presion_sistolica",
    "diaBP": "presion_diastolica", "BMI": "imc", "heartRate": "frecuencia_cardiaca",
    "glucose": "glucosa", "TenYearCHD": "chd_10_anios",
}

datos_originales = pd.read_csv(RUTA_DATOS)
datos = datos_originales.rename(columns=NOMBRES_EN_ESPANOL)

assert datos.columns.size == len(NOMBRES_EN_ESPANOL), "Faltan columnas por renombrar"
print(f"Filas: {datos.shape[0]:,} | Columnas: {datos.shape[1]}")
datos.head()

Filas: 4,238 | Columnas: 16


,sexo_masculino,edad,educacion,fumador_actual,cigarrillos_dia,medicamento_presion,acv_previo,hipertension,diabetes,colesterol_total,presion_sistolica,presion_diastolica,imc,frecuencia_cardiaca,glucosa,chd_10_anios
0,1,39,4.000,0,0.000,0.000,0,0,0,195.000,106.000,70.000,26.970,80.000,77.000,0
1,0,46,2.000,0,0.000,0.000,0,0,0,250.000,121.000,81.000,28.730,95.000,76.000,0
2,1,48,1.000,1,20.000,0.000,0,0,0,245.000,127.500,80.000,25.340,75.000,70.000,0
3,0,61,3.000,1,30.000,0.000,0,1,0,225.000,150.000,95.000,28.580,65.000,103.000,1
4,0,46,3.000,1,23.000,0.000,0,0,0,285.000,130.000,84.000,23.100,85.000,85.000,0


El conjunto contiene **4.238 personas y 16 variables**. Cada fila describe a una persona al inicio del seguimiento
(edad, presión, colesterol, hábitos y diagnósticos) e indica si desarrolló enfermedad coronaria en los 10 años
siguientes (`chd_10_anios`). Como cada registro mide los factores antes de que ocurra o no la enfermedad, los datos
permiten estudiar qué características previas se asocian al riesgo futuro.